# Lab 4: Cross-Validation and the Bootstrap

**CS 463 · Due Monday, Oct 26, 8:30 AM in Canvas**

Based on §5.3 (Lab: Cross-Validation and the Bootstrap) of James, Witten, Hastie, Tibshirani, and
Taylor, *An Introduction to Statistical Learning with Applications in Python* (Springer, 2023),
using the `Auto` data from the `ISLP` package.

This lab takes the two Chapter 5 notebooks and does the same things with real data and the tools
you'd use on Kaggle: [Notebook 1](../../Course_Documents/Slides/ch_5/ch_5_01_knn_bayes_boundary.ipynb) (the validation set approach and cross-validation) and
[Notebook 2](../../Course_Documents/Slides/ch_5/ch_5_demos_2_bootstrap.ipynb) (the bootstrap). Keep them open beside the lab; most prompts point back to a
section of one of them. There's no new theory here.

### Overview

You'll predict a car's `mpg` from its `horsepower` with KNN, choose $K$, then measure how much
to trust the numbers you get.

0. **Part 0, the data**: a quick look at `Auto`, using what you practiced in Lab 1.
1. **Part 1, the validation set approach** (Notebook 1, §8): choose $K$ with one 50/50 split,
   see how much the answer changes with a different split, and see why the split must be random.
2. **Part 2, 10-fold cross-validation** (Notebook 1, §9–11): choose $K$ again, see why the result
   is more stable, refit the final model, and find the bug in hypothetical code.
3. **Part 3, the bootstrap for one number** (Notebook 2, §2): write a bootstrap loop to get a
   standard error for the median `mpg`.
4. **Part 4, the bootstrap for a regression slope** (Notebook 2, §2): bootstrap a fitted slope,
   and compare with the standard error from the regression summary.
5. **Part 5, better or just lucky?** (Notebook 2, §3–4): bootstrap the difference between two
   models' errors, decide which gaps are real, and pick a model to deploy.

At the end, you'll see how the bootstrap is usually run in practice: one call to SciPy.

### How this lab works

- Work through the parts in order.
- Everything you turn in is labeled by part number and letter, like **2(a)**, and marked as
  either **Code** or **Written**:
  - **Code:** write your code in the cell right below, which starts with numbering like `# 2(a)` and
    `# YOUR CODE HERE`. Some of these cells are partly written for you: fill in the part marked
    `# YOUR CODE HERE`, and leave the rest.
  - **Written:** replace `YOUR ANSWER HERE` in the **2(b) Answer:** cell right below. Written
    answers: 1–3 sentences.
- Most prompts have collapsible **Hints**. Open them whenever you're stuck.
- The **Stretch** item at the end is optional and not graded.
- Anything else, like "Run this cell," is just an instruction. There's nothing to turn in.

**What you're responsible for:** the validation set approach, 10-fold cross-validation, choosing
$K$ and refitting, the bootstrap standard error, and the percentile interval for a difference.
The note in Part 4 on *why* formula standard errors can be off is **awareness level**.

**Checklist** (18 required items):

| Part | Code | Written |
|---|---|---|
| 0 | 0(a), 0(b) | 0(c) |
| 1 | 1(a) | 1(b), 1(c), 1(d) |
| 2 | 2(a), 2(c) | 2(b), 2(d) |
| 3 | 3(a) | 3(b) |
| 4 | 4(a) | 4(b) |
| 5 | 5(a) | 5(b), 5(c) |

- Keep `ch_5_lab_helpers.py` in the same folder as this notebook (on Colab, upload it with the
  folder icon in the left sidebar). It holds the plotting code, and you don't need to read it.
  Don't turn it in.
- Before submitting, run **Restart & Run All** and make sure every cell runs.

**Note:** If you just run everything right now, you'll get some errors, because some of the
cells I've provided depend on cells you complete first. That's expected.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from ISLP import load_data

from ch_5_lab_helpers import (validation_curves, cv_curves, plot_knn_fits, plot_validation_curves,
                              plot_cv_vs_validation, plot_bootstrap, plot_line_fit)

## Part 0: The data

You explored `Auto` in Lab 1. In this lab, we'll predict a car's `mpg` from its `horsepower`.
Before modeling, take a quick look at the two columns we'll use, and at the row labels.

Run this cell to load the data and preview it.

In [ ]:
Auto = load_data('Auto')
Auto.head()

**0(a) · Code.** Show the dimensions (num rows and columns) of `Auto` and the type of each column. Then show summary statistics
(count, mean, min, max, and so on) for just `mpg` and `horsepower`.

<details>
<summary>Hint</summary>

You did all three in Lab 1, Exercise 8. To summarize only some columns, select them first with a
list of column names, then summarize the result.

</details>

In [ ]:
# 0(a)
### YOUR CODE HERE


**0(b) · Code.** The row labels (the index) are car names. Set `n_repeated_names` to the number of
rows whose name already appeared in an earlier row.

<details>
<summary>Hint</summary>

Lab 1's 9(d) check used `.duplicated()` on the index. It marks each label that already appeared
above as `True`. On made-up labels:

```python
labels = pd.Index(['ann', 'bo', 'ann', 'cy', 'bo'])
print(labels.duplicated())
```

```
[False False  True False  True]
```

Summing a `True`/`False` array counts the `True`s.

</details>

In [ ]:
# 0(b)
### YOUR CODE HERE
print(n_repeated_names, 'rows reuse a car name from an earlier row')

**0(c) · Written.** Answer briefly, using your output from 0(a) and 0(b):

1. What does one row of `Auto` represent, and how many are there?
2. What range of `horsepower` do the cars cover? In Part 2 you'll predict `mpg` for cars with 100
   and 200 horsepower. Would you trust a prediction for a 300-horsepower car? Why or why not?
3. Later, you'll pick rows at random. Why is picking them by *label* (`.loc`) risky with these data?
   (Lab 1's 9(d) ran into the same problem.)

**0(c) Answer:**

_YOUR ANSWER HERE_


Run this cell to set up the model's input and output. scikit-learn wants `X` to be 2-D (a table
with one column per input), even when there's only one input, so we select `horsepower` with a
list of names.

In [ ]:
X = Auto[['horsepower']]     # a DataFrame with one column
y = Auto['mpg']
X.shape, y.shape

### KNN for a number

In [Notebook 1](../../Course_Documents/Slides/ch_5/ch_5_01_knn_bayes_boundary.ipynb), KNN *classified* a point by a vote of its $K$ nearest training points.
To predict a **number**, like `mpg`, KNN does the same search and **averages** the neighbors'
values instead of voting (ISL §3.5). scikit-learn calls this `KNeighborsRegressor`. It has the same
`fit` and `predict` methods as `KNeighborsClassifier`, and $K$ is still the flexibility knob:
small $K$ is flexible, and large $K$ is rigid.

Run this cell to see three fits. *(You don't need to understand how this plot was generated.)*

In [ ]:
plot_knn_fits(X, y, K_values=(1, 20, 150));

We'll measure how good a fit is with the **mean squared error** (MSE): the average of
(actual `mpg` − predicted `mpg`)$^2$ over the cars we score.

$$
\mathrm{MSE} = \frac{1}{m}\sum_{i=1}^{m}\left(y_i - \hat{y}_i\right)^2
$$

where $m$ is the number of cars we score, $y_i$ is car $i$'s actual `mpg`, and $\hat{y}_i$ is the
model's predicted `mpg` for it. Lower is better. It's in squared mpg, so an MSE of 18 means predictions are typically off by about $\sqrt{18} \approx 4$ mpg.

Which $K$ predicts *new* cars best? We can't collect new cars, so we hold some back.

## Part 1: The validation set approach

**Points back to:** [Notebook 1, §8 and Demo F](../../Course_Documents/Slides/ch_5/ch_5_01_knn_bayes_boundary.ipynb#demo-f).

**Overview:** Split the 392 cars in half **at random**: each car lands in exactly one half, and every car is
equally likely to land in either. If you coded this by hand, you might shuffle the rows, then split the dataset in half. Instead, use a tool (see the hint below) that makes this step easy. Then, fit on one half (the
training set), and measure the MSE on the other half (the validation set), once for each $K$. Then
choose the $K$ with the lowest validation MSE.

**1(a) · Code.** Split `X` and `y` in half with `train_test_split`, using `test_size=0.5` and
`random_state=0`. Then, for each $K$ in `K_values`, fit `KNeighborsRegressor(n_neighbors=K)` on the
training half, and append its MSE on the validation half to a list named `valid_mse`. The cell's
last line prints the $K$ you'd choose.

<details>
<summary>Hint 1: the tools</summary>

- `train_test_split(X, y, test_size=0.5, random_state=0)` returns **four** new pieces, in this
  order: the training `X`, the validation `X`, the training `y`, and the validation `y`. It doesn't
  mutate `X` or `y`: it shuffles a list of row *positions*, cuts it, and copies those rows out.
  So the pieces come back in shuffled order, and each row keeps its original label.
  `random_state` makes the shuffle repeatable.
- It shuffles **by default**, so you don't shuffle first yourself. (`KFold`, in
  Part 2, does *not* shuffle unless you pass `shuffle=True`.)
- `mean_squared_error(actual, predicted)` returns one number: the MSE.
- A common mistake: scoring on the training half. The MSE must use the rows the model didn't see.
- Notebook 1's code above Demo F does the same loop for KNN classification, with error = 1 −
  accuracy instead of MSE.

</details>

<details>
<summary>Hint 2: an example on made-up data</summary>

Eight students' study hours and exam scores, a 6/2 split, and a silly "model" that always guesses
the training average:

```python
hours = pd.DataFrame({'hours': [1, 2, 3, 4, 5, 6, 7, 8]})
score = pd.Series([52, 55, 61, 64, 70, 71, 78, 83])
h_train, h_valid, s_train, s_valid = train_test_split(hours, score, test_size=0.25, random_state=1)
print(len(h_train), 'training rows,', len(h_valid), 'validation rows')
guess = [s_train.mean()] * len(s_valid)          # always guess the training average
print(mean_squared_error(s_valid, guess))
```

```
6 training rows, 2 validation rows
170.0
```

Your version replaces the silly guess with a KNN model's `predict` on the validation `X`, inside a
loop over `K_values`.

</details>

In [ ]:
# 1(a)
K_values = [1, 2, 5, 10, 20, 30, 50, 75, 100, 150]

### YOUR CODE HERE

print('Chosen K:', K_values[np.argmin(valid_mse)])

Run this cell to plot your validation curve. *(You don't need to understand how this plot was
generated.)*

In [ ]:
plot_validation_curves(K_values, [valid_mse], title='Validation set approach, one split');

**1(b) · Written.** *Before you run the next cell:* if we split the cars with a different
`random_state`, would we get the same curve? The same chosen $K$? Any prediction is fine; you'll
check it next.

**1(b) Answer:**

_YOUR ANSWER HERE_


Run this cell. `validation_curves` (in the helpers file) repeats a loop like your wrote in 1(a) for 10 different
splits, `random_state` 0 through 9, and the table lists the $K$ each split would choose.

In [ ]:
valid_runs = validation_curves(X, y, K_values, seeds=range(10))
plot_validation_curves(K_values, valid_runs, title='Validation set approach, 10 random splits')
pd.DataFrame({'chosen K': [K_values[np.argmin(c)] for c in valid_runs],
              'lowest validation MSE': [round(min(c), 1) for c in valid_runs]},
             index=pd.Index(range(10), name='random_state'))

**1(c) · Written.** Look at the plot and the table. If a teammate chose $K$, and reported its
MSE, from a single 50/50 split, how much would you trust those two numbers? Cite what you see.

**1(c) Answer:**

_YOUR ANSWER HERE_


### Why shuffle?

What if we skipped the shuffle and just cut the data in the middle? Run this cell. It trains KNN
with $K = 30$ on the first 196 rows and scores it on the last 196, then compares with your shuffled
split from 1(a). *(Code worth reading.)*

In [ ]:
first_half, second_half = train_test_split(Auto, test_size=0.5, shuffle=False)   # rows stay in order
model = KNeighborsRegressor(n_neighbors=30).fit(first_half[['horsepower']], first_half['mpg'])
unshuffled_mse = mean_squared_error(second_half['mpg'], model.predict(second_half[['horsepower']]))

print(f'At K = 30: \n Validation MSE Unshuffled = {unshuffled_mse:.1f}, \n'
      f' Validation MSE Shuffled = {valid_mse[K_values.index(30)]:.1f} from 1(a)')
pd.DataFrame({'model years': [f"19{half['year'].min()}–19{half['year'].max()}" for half in (first_half, second_half)],
              'mean mpg': [round(half['mpg'].mean(), 1) for half in (first_half, second_half)]},
             index=['training half (first 196 rows)', 'validation half (last 196 rows)'])

**1(d) · Written.** Why is the unshuffled validation MSE so much worse? Use the table. Which idea from
[Notebook 2, §4](../../Course_Documents/Slides/ch_5/ch_5_demos_2_bootstrap.ipynb) is this a small
version of?

**1(d) Answer:**

_YOUR ANSWER HERE_


## Part 2: 10-fold cross-validation

**Points back to:** [Notebook 1, §9–11, Demos G1 and G2](../../Course_Documents/Slides/ch_5/ch_5_01_knn_bayes_boundary.ipynb#demo-g1).

Instead of one split, cut the 392 cars into 10 folds. Each fold takes one turn as the validation
set while the model trains on the other 9, and we average the 10 MSEs. Every car is used for
validation exactly once.

**2(a) · Code.** For each $K$ in `K_values`, compute the 10-fold cross-validation MSE of
`KNeighborsRegressor(n_neighbors=K)` on all of `X` and `y`, and append it to a list named `cv_mse`.
Use `folds = KFold(n_splits=10, shuffle=True, random_state=0)` for every $K$, so every $K$ is
scored on the same folds.

<details>
<summary>Hint 1: the tools</summary>

- `KFold(n_splits=10, shuffle=True, random_state=0)` doesn't mutate the data. It's a *recipe* for
  cutting rows into 10 shuffled folds, which you pass to `cross_val_score` as `cv=`. Without
  `shuffle=True`, each fold would be a block of consecutive rows (see 1(d) for why that's bad here).
- `cross_val_score(model, X, y, cv=folds, scoring='neg_mean_squared_error')` fits a fresh copy
  of `model` once per fold and returns an array of 10 scores, one per fold.
- **The sign.** The `scoring=` options in scikit-learn's model-selection tools (`cross_val_score`,
  and others you'll meet later) all follow one rule: higher is better. So
  `'neg_mean_squared_error'` reports $-\mathrm{MSE}$ for each fold. Metric functions like the
  `mean_squared_error` you used in 1(a) return the MSE itself. Negate the 10 scores to get MSEs,
  then average them.
- Notebook 1, §10, has this loop for KNN classification (with accuracy, the default score).

</details>

<details>
<summary>Hint 2: an example on made-up data</summary>

The study-hours data from 1(a), with linear regression and 4 folds:

```python
folds_demo = KFold(n_splits=4, shuffle=True, random_state=1)
scores = cross_val_score(LinearRegression(), hours, score, cv=folds_demo,
                         scoring='neg_mean_squared_error')
print(scores)            # one (negative) MSE per fold
print(-scores.mean())    # the cross-validation MSE
```

```
[-1.73469388 -0.71898241 -1.59750567 -4.88228957]
2.233367882727155
```

</details>

In [ ]:
# 2(a)
folds = KFold(n_splits=10, shuffle=True, random_state=0)

### YOUR CODE HERE

print('Chosen K:', K_values[np.argmin(cv_mse)])

Does cross-validation agree with itself better than the validation set approach did? Run this
cell. `cv_curves` (in the helpers file) repeats your 2(a) loop for 10 different shuffles of the
cars into folds, `random_state` 0 through 9. On the left are Part 1's 10 validation-set curves,
on the same vertical scale. *(You don't need to understand how this plot was generated.)*

In [ ]:
cv_runs = cv_curves(X, y, K_values, seeds=range(10))
plot_cv_vs_validation(K_values, cv_runs, valid_runs)
pd.DataFrame({'chosen K (validation set)': [K_values[np.argmin(c)] for c in valid_runs],
              'chosen K (10-fold CV)': [K_values[np.argmin(c)] for c in cv_runs]},
             index=pd.Index(range(10), name='random_state'))

**2(b) · Written.** Why do the cross-validation curves agree with each other so much more than
the validation-set curves? The chosen $K$ still varies a little from shuffle to shuffle. Should
that worry you? Which values of $K$ look reasonable?

**2(b) Answer:**

_YOUR ANSWER HERE_


**2(c) · Code.** Cross-validation is for *choosing*. Each of its models trained on only about 353
cars. Set `best_K` to the $K$ your `cv_mse` chose, fit `final_model`, a `KNeighborsRegressor` with
that $K$, on **all** 392 cars, and print its predicted `mpg` for cars with 100 and 200 horsepower.
([Notebook 1, §11](../../Course_Documents/Slides/ch_5/ch_5_01_knn_bayes_boundary.ipynb#section-11))

<details>
<summary>Hint</summary>

`predict` wants the same kind of `X` the model was fit on: a DataFrame with a `horsepower` column.
For a model fit on a column named `hours`:

```python
m = KNeighborsRegressor(n_neighbors=3).fit(hours, score)
print(m.predict(pd.DataFrame({'hours': [2.5, 9]})))
```

```
[56.         77.33333333]
```

</details>

In [ ]:
# 2(c)
### YOUR CODE HERE


**2(d) · Written.** *Find the bug.* Why isn't `reported_error`, defined below, a reliable estimate of the model's error on new/unseen cars?"

```python
# Hold back a quarter of the cars as a test set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)

# Score every K on the test set
test_mse = []
for K in K_values:
    model = KNeighborsRegressor(n_neighbors=K)
    model.fit(X_train, y_train)
    test_predictions = model.predict(X_test)
    test_mse.append(mean_squared_error(y_test, test_predictions))

# Keep the K with the lowest test MSE, and report that MSE
best_position = np.argmin(test_mse)
chosen_K = K_values[best_position]
reported_error = test_mse[best_position]
```

What's wrong with the claim? What should they do instead? (See [Notebook 2](../../Course_Documents/Slides/ch_5/ch_5_demos_2_bootstrap.ipynb)'s note on
validation versus test sets, and the Kaggle paragraph in [Notebook 1, §11](../../Course_Documents/Slides/ch_5/ch_5_01_knn_bayes_boundary.ipynb#section-11).)

**2(d) Answer:**

_YOUR ANSWER HERE_


## Part 3: The bootstrap for one number

**Points back to:** [Notebook 2, §2 and Demo B](../../Course_Documents/Slides/ch_5/ch_5_demos_2_bootstrap.ipynb).

The median `mpg` of our 392 cars is one **estimate**, $\hat\theta$. A different sample of 392
cars would give a different median. The bootstrap estimates how different, using only the sample
we have:

1. Draw a **bootstrap sample**: $n$ rows from the data, at random, **with replacement**.
2. Compute the statistic on it, $\hat\theta^{*b}$.
3. Repeat $B$ times.
4. The standard deviation of the $B$ values is the **bootstrap standard error**,
   $\mathrm{SE}_B(\hat\theta)$.

**3(a) · Code.** Complete the loop: draw $B = 2{,}000$ bootstrap samples of `mpg`, and append each
one's median to `boot_medians`. Then set `se_median` to the bootstrap standard error.

<details>
<summary>Hint 1: the tools</summary>

- `rng.integers(0, n, size=n)` returns `n` random whole numbers from `0` to `n - 1`, with repeats
  allowed: that's "with replacement". Use them as positions.
- Indexing a NumPy array with an array of positions, `values[positions]`, returns those elements,
  repeats included. That's one bootstrap sample.
- `np.median(...)` is the median. `np.std(values, ddof=1)` is the standard deviation with the
  $B - 1$ in the formula. (`ddof=0`, the default, divides by $B$ instead.)
- Demo B's cell under **Code worth reading** does this for the mean and the median of 40 waits.

</details>

<details>
<summary>Hint 2: one bootstrap sample, on made-up data</summary>

```python
rng_demo = np.random.default_rng(1)   # create a random number generator with the seed, 1.
heights = np.array([160, 172, 168, 181, 175])
picks = rng_demo.integers(0, len(heights), size=len(heights))
print(picks)              # 5 positions, with repeats
print(heights[picks])     # the bootstrap sample
```

```
[2 2 3 4 0]
[168 168 181 175 160]
```

Your loop does this $B$ times with `rng` and `mpg`, and keeps each sample's median.

</details>

In [ ]:
# 3(a)
mpg = Auto['mpg'].to_numpy()
n, B = len(mpg), 2_000
B_median = B                  # saved for the SciPy comparison at the end of the lab
rng = np.random.default_rng(463) # create a random number generator with seed 463 (for reproducible results)
boot_medians = []

### YOUR CODE HERE

print(f'median mpg: {np.median(mpg):.2f}    bootstrap standard error: {se_median:.2f}')

Run this cell to see the $B$ bootstrap medians. *(You don't need to understand how this plot was
generated.)*

In [ ]:
plot_bootstrap(boot_medians, np.median(mpg), xlabel='median mpg of a bootstrap sample',
               title=f'{B:,} bootstrap medians');

**3(b) · Written.** In two or three sentences, and in mpg, what does `se_median` tell you? Then give
the rough 95% range for the median, $\hat\theta \pm 2\,\mathrm{SE}_B(\hat\theta)$
([Notebook 2, §2](../../Course_Documents/Slides/ch_5/ch_5_demos_2_bootstrap.ipynb)).

**3(b) Answer:**

_YOUR ANSWER HERE_


## Part 4: The bootstrap for a regression slope

**Points back to:** [Notebook 2, §2](../../Course_Documents/Slides/ch_5/ch_5_demos_2_bootstrap.ipynb), and the Lab 2 (Chapter 3) summary table.

The bootstrap works for almost any number computed from data, including a fitted coefficient.
Here's the least squares line from Lab 2's toolkit, predicting `mpg` from `horsepower`. Its
summary already reports a **formula** standard error for the slope (`bse`). We'll get a second
one from the bootstrap and compare.

In [ ]:
line_model = smf.ols('mpg ~ horsepower', data=Auto).fit()
print(f"slope: {line_model.params['horsepower']:.4f}    formula SE: {line_model.bse['horsepower']:.4f}")
plot_line_fit(Auto, 'horsepower', 'mpg', line_model);

**4(a) · Code.** Bootstrap the slope with $B = 1{,}000$: each round, draw a bootstrap sample of
the **rows** of `Auto`, refit `smf.ols('mpg ~ horsepower', ...)` on it, and append the
`horsepower` coefficient to `boot_slopes`. Then set `se_slope_boot` to the bootstrap standard
error. This cell takes a few seconds.

<details>
<summary>Hint 1: the tools</summary>

- This is 3(a)'s loop, with two changes: the sample is a set of **rows** of a DataFrame, and the
  statistic is a refit model's coefficient (`.params['horsepower']`, as in Lab 2).
- **Pick rows by position with `.iloc`, not by label with `.loc`.** `Auto`'s labels are car names,
  and some repeat (you counted them in 0(b)). `.loc` would return *every* row with a drawn name, so the bootstrap sample
  would have the wrong size.

</details>

<details>
<summary>Hint 2: `.loc` vs. `.iloc`, on made-up data</summary>

```python
pets = pd.DataFrame({'weight': [4, 30, 5, 28]}, index=['cat', 'dog', 'cat', 'dog'])
print(len(pets.loc[['cat']]))    # every row labeled 'cat'
print(len(pets.iloc[[0]]))       # only the row at position 0
print(pets.iloc[[0, 0, 3]])      # positions can repeat
```

```
2
1
     weight
cat       4
cat       4
dog      28
```

</details>

In [ ]:
# 4(a)
n, B = len(Auto), 1_000
rng = np.random.default_rng(463)
boot_slopes = []

### YOUR CODE HERE

print(f"formula SE:   {line_model.bse['horsepower']:.4f}")
print(f'bootstrap SE: {se_slope_boot:.4f}')
plot_bootstrap(boot_slopes, line_model.params['horsepower'], xlabel='slope of a bootstrap sample',
               title=f'{B:,} bootstrap slopes');

**Why can the two disagree? *(awareness level)*** The formula standard error from Lab 2 rests
on assumptions about the model. The biggest one here: it assumes the **straight-line model is
right**, so that the scatter around the line is just noise. When the true relationship curves,
the formula's estimate of that noise, and with it the standard error, is off. The bootstrap makes
no such assumption: it just refits on resampled data and looks at the spread (ISL §5.3.4).

**4(b) · Written.** The two standard errors differ by about 15%. Using the scatter plot above and
this note, which would you trust more for this data, and why? Do they agree about whether
`mpg` is related to `horsepower`?

**4(b) Answer:**

_YOUR ANSWER HERE_


## Part 5: Better, or just lucky?

**Points back to:** [Notebook 2, §3 and Demo C](../../Course_Documents/Slides/ch_5/ch_5_demos_2_bootstrap.ipynb).

We hold back 100 cars and fit three models on the other 292: linear regression, and KNN with
$K = 20$ and $K = 50$. Then we record each model's **squared error on each held-out car**. A
model's MSE is the average of its 100 squared errors.

Run this cell. *(Code worth reading.)*

In [ ]:
X_fit, X_held, y_fit, y_held = train_test_split(X, y, test_size=100, random_state=0)

models = {'linear': LinearRegression(),
          'K = 20': KNeighborsRegressor(n_neighbors=20),
          'K = 50': KNeighborsRegressor(n_neighbors=50)}
sq_err = {}
for name, model in models.items():
    model.fit(X_fit, y_fit)
    sq_err[name] = ((y_held - model.predict(X_held)) ** 2).to_numpy()   # one squared error per held-out car

pd.DataFrame({'MSE on the 100 held-out cars': [errors.mean() for errors in sq_err.values()]},
             index=pd.Index(sq_err.keys(), name='model')).round(2)

KNN with $K = 50$ has the lowest MSE on these 100 held-out cars. Is either gap real, or the luck of which
100 cars were held out? As in Demo C, we bootstrap the **difference** in MSE, $\hat{\theta}$: resample the 100
held-out cars, and recompute both models' MSEs on the **same** resampled cars each round. Then, we compute the difference, 

$ \hat{\theta} = \text{MSE}_{\text{model A}} - \text{MSE}_{\text{model B}} $

**5(a) · Code.** Complete the function, `bootstrap_mse_difference`. Each of the `B` rounds draws one set of
positions `idx` and appends (model A's MSE on those cars) − (model B's MSE on those cars) to
`diffs`. After the loop, set `low` and `high` to the 95% percentile interval: the 2.5th and
97.5th percentiles of `diffs`. Uncomment the "sample calls" at the bottom of the cell to help with debugging. Comment them out again when you're finished (or delete them).

<details>
<summary>Hint 1: the tools</summary>

- Use **one** `idx` per round for both models, so each difference compares them on the same cars.
  That's what makes the comparison meaningful.
- `sq_err_a[idx].mean()` is model A's MSE on the resampled cars.
- `np.percentile(values, [p1, p2])` returns two numbers: the `p1`th and `p2`th percentiles.

</details>

<details>
<summary>Hint 2: examples on made-up data</summary>

```python
toy = np.array([3, 9, 1, 7, 5, 2, 8, 6, 4, 10])
print(np.percentile(toy, [10, 90]))

a = np.array([1.0, 4.0, 9.0])
b = np.array([2.0, 2.0, 2.0])
idx = np.array([2, 0, 2])                  # the same positions for both arrays
print(a[idx], b[idx], a[idx].mean() - b[idx].mean())
```

```
[1.9 9.1]
[9. 1. 9.] [2. 2. 2.] 4.333333333333333
```

</details>

In [ ]:
# 5(a)
def bootstrap_mse_difference(sq_err_a, sq_err_b, B=2_000, seed=463):
    """Bootstrap (model A's MSE) - (model B's MSE) on the same held-out cars.
    Returns the list of B differences and the 95% percentile interval (low, high)."""
    rng = np.random.default_rng(seed)
    n = len(sq_err_a)
    diffs = []
    ### YOUR CODE HERE
    return diffs, low, high

# Sample calls to debug your function. The expected output is in the comments on the print lines.
# Identical models never differ, so every difference is 0:

# Sample calls 1
# same = np.array([4.0, 9.0, 1.0, 16.0])
# diffs, low, high = bootstrap_mse_difference(same, same, B=5)
# print(np.array(diffs))                 # [0. 0. 0. 0. 0.]
# print(low, high)                       # 0.0 0.0

# Sample calls 2
# If model A's squared error is always 5 and model B's is always 2, A's MSE is always 3 higher:

# five = np.array([5.0, 5.0, 5.0, 5.0])
# two = np.array([2.0, 2.0, 2.0, 2.0])
# diffs, low, high = bootstrap_mse_difference(five, two, B=5)
# print(np.array(diffs))                 # [3. 3. 3. 3. 3.]
# print(low, high)                       # 3.0 3.0


# Sample calls 3
# Swapping the models flips the sign. B controls how many differences come back:

# five = np.array([5.0, 5.0, 5.0, 5.0])
# two = np.array([2.0, 2.0, 2.0, 2.0])
# diffs, low, high = bootstrap_mse_difference(two, five, B=10)
# print(len(diffs))                      # 10
# print(low, high)                       # -3.0 -3.0

Run this cell to bootstrap two comparisons and plot them. *(You don't need to understand how the
plot was generated.)*

In [ ]:
comparisons = [('linear', 'K = 20'), ('K = 50', 'K = 20')]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (a, b) in zip(axes, comparisons):
    diffs, low, high = bootstrap_mse_difference(sq_err[a], sq_err[b])
    observed = sq_err[a].mean() - sq_err[b].mean()
    print(f'{a} minus {b}: observed {observed:+.2f}, 95% interval from {low:+.2f} to {high:+.2f}')
    plot_bootstrap(diffs, observed, xlabel=f'MSE({a}) - MSE({b})', title=f'{a} minus {b}',
                   interval=(low, high), zero_line=True, ax=ax)
fig.tight_layout()

**Reading these plots.** In each title, model A is named first and model B second: in "linear
minus K = 20", model A is linear regression and model B is KNN with $K = 20$. The horizontal axis
shows the difference (as in *subtraction*)

$$
\mathrm{MSE}(\text{model A}) - \mathrm{MSE}(\text{model B}),
$$

in squared mpg, computed on one bootstrap sample of the 100 held-out cars. Each bar's height is the
number of bootstrap samples whose difference landed in that bar's range. The solid black line is the
difference on the original 100 held-out cars.

- **Positive** (right of the dotted line at 0): model A's MSE is bigger, so model A predicted
  **worse** on those cars.
- **Negative** (left of 0): model A's MSE is smaller, so model A predicted **better**.
- **Zero**: Think about what this result means. 

**5(b) · Written.** For each comparison, is the gap in MSE real, or could it be luck? Which model
would you deploy, and how sure are you? Cite the intervals.

**5(b) Answer:**

_YOUR ANSWER HERE_


**5(c) · Written.** These cars are from 1970 to 1982. An engineer wants to use your final model from
2(c), and its cross-validation MSE, for cars sold today. Should they trust that MSE? Use the
vocabulary from [Notebook 2, §4](../../Course_Documents/Slides/ch_5/ch_5_demos_2_bootstrap.ipynb).

**5(c) Answer:**

_YOUR ANSWER HERE_


**Something to think about** *(optional, nothing to turn in).* [Notebook 2, §4](../../Course_Documents/Slides/ch_5/ch_5_demos_2_bootstrap.ipynb)
says cross-validation and the bootstrap assume each row was drawn **independently**. In `Auto`,
many models appear in several model years (you counted the repeated names in 0(b)), and a model
barely changes from one year to the next. Does that assumption fully hold here? If not, how might
it affect the estimates in this lab?

<details>
<summary>Our take</summary>

Not fully. Two rows for the same model in back-to-back years are near-copies, so they aren't
independent draws. In cross-validation, one can land in the training folds while its near-copy is
in the validation fold, so the model gets scored on cars it really has some information about, and the error
estimate can come out a little too optimistic. The bootstrap treats near-copies as separate
pieces of evidence, so its standard errors can come out a little too small. A common fix is to
keep all rows for a model together, always in the same fold or resampled as a group, but that's
beyond this course.

</details>

## In practice: SciPy's bootstrap

You wrote the bootstrap loop yourself in Parts 3–5, so you know exactly what it does. In practice,
you'll usually call a library function instead. SciPy's
[`scipy.stats.bootstrap`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.bootstrap.html)
runs the same resample-and-recompute loop. The documentation page lists everything it can do.

Run this cell. It bootstraps the median `mpg`, with the same seed and $B$ as 3(a), and compares
the results with yours. *(Code worth reading.)*

- The data go in as a **tuple** of arrays: `(mpg,)`. The trailing comma makes a one-element tuple.
  (Statistics that need two arrays get two.)
- The statistic is a function, `np.median`, passed without calling it. `bootstrap` calls it once
  per bootstrap sample.
- `n_resamples` is $B$, and `rng` is a random number generator, like `rng` in 3(a).
- `method='percentile'` asks for the percentile interval this course uses. SciPy's default is an
  adjusted interval we don't cover.
- The returned object has `.standard_error`, `.confidence_interval` (with `.low` and `.high`), and
  `.bootstrap_distribution`: the $B$ bootstrap medians themselves.

In [ ]:
from scipy.stats import bootstrap

result = bootstrap((mpg,), np.median, n_resamples=B_median, method='percentile',
                   rng=np.random.default_rng(463))
print(f'SciPy: SE {result.standard_error:.2f}, 95% interval {result.confidence_interval.low:.2f} '
      f'to {result.confidence_interval.high:.2f}')
print(f'3(a):  SE {se_median:.2f}, 95% interval {np.percentile(boot_medians, 2.5):.2f} '
      f'to {np.percentile(boot_medians, 97.5):.2f}')

The numbers should match exactly: with the same seed, SciPy draws the same bootstrap samples your loop did.

## Wrap-up

- One validation split gives a noisy estimate, and a noisy choice of $K$ (Part 1).
- 10-fold cross-validation averages over folds and uses every point for validation, so it agrees
  with itself and reliably rules out bad $K$ values; then refit on all the data (Part 2).
- The bootstrap gives a standard error for almost any number computed from data, a median or a
  fitted slope, with the same short loop (Parts 3–4). In practice, `scipy.stats.bootstrap` runs
  that loop for you.
- Before trusting "model A beat model B", bootstrap the difference: if the interval includes 0,
  the data can't tell them apart (Part 5).
- All of it assumes new data from the same data-generating process (Part 5(c)).

*Good place to stop.* The stretch item below is optional.

**Stretch 1 · Code.** *(Optional.)* Leave-one-out cross-validation is $k$-fold with $k = n$: 392
folds of one car each ([Notebook 1, Demo G1](../../Course_Documents/Slides/ch_5/ch_5_01_knn_bayes_boundary.ipynb#demo-g1)). Compute the LOOCV MSE for each $K$
in `K_values` into `loo_mse`, time it with `time.perf_counter()`, and compare its chosen $K$ and
its curve with 10-fold's `cv_mse`. Which would you use on a dataset with a million rows?

<details>
<summary>Hint</summary>

`LeaveOneOut` lives in `sklearn.model_selection`, next to `KFold`. It takes no arguments, and an
instance of it goes where 2(a) passed `folds`. `time.perf_counter()` returns a time in seconds:
call it before and after, and subtract.

</details>

In [ ]:
# Stretch 1
### YOUR CODE HERE
